# 01 · Exploratory data analysis

Engine-health snapshot readings: class balance, distributions, correlations, missing values and outliers.
Run `python scripts/download_data.py` first. Plots are saved to `docs/img/` for the README.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.data import dataset_info, load_dataset
from ml.features import SENSOR_COLUMNS, SENSOR_LABELS

ROOT = Path.cwd() if (Path.cwd() / 'ml').exists() else Path.cwd().parent
DATA = ROOT / 'data' / 'raw' / 'engine_data.csv'
IMG = ROOT.parent / 'docs' / 'img'
IMG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

df = load_dataset(DATA)
print(dataset_info(DATA))
df.describe().T

## Class balance and missing values

In [ ]:
counts = df['fault'].value_counts().sort_index()
print(counts)
print('fault share:', round(df['fault'].mean(), 3))
print('missing values per column:')
print(df[SENSOR_COLUMNS].isna().sum())
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(['healthy (0)', 'faulty (1)'], counts.values, color=['#64748b', '#1f63d8'])
ax.set_ylabel('rows'); ax.set_title('Class balance')
fig.tight_layout(); fig.savefig(IMG / 'class_balance.png')

## Distributions by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, col in zip(axes.flat, SENSOR_COLUMNS):
    for label, colour in ((0, '#64748b'), (1, '#1f63d8')):
        ax.hist(df.loc[df.fault == label, col], bins=40, alpha=0.55, color=colour, label='faulty' if label else 'healthy', density=True)
    ax.set_title(SENSOR_LABELS[col], fontsize=10)
axes.flat[0].legend()
fig.tight_layout(); fig.savefig(IMG / 'distributions.png')

## Correlations

In [ ]:
corr = df[SENSOR_COLUMNS + ['fault']].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
labels = [SENSOR_LABELS.get(c, c) for c in corr.columns]
ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(labels)), labels, fontsize=8)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, shrink=0.8); ax.set_title('Pearson correlation')
fig.tight_layout(); fig.savefig(IMG / 'correlations.png')

## Outliers (share of rows outside the 1st/99th percentile, which preprocessing clips)

In [ ]:
low, high = df[SENSOR_COLUMNS].quantile(0.01), df[SENSOR_COLUMNS].quantile(0.99)
outside = ((df[SENSOR_COLUMNS] < low) | (df[SENSOR_COLUMNS] > high)).mean().round(4)
outside.rename('share clipped')

## Notes

- Write down what you see here (overlap between classes, which sensors separate them, skew) — it explains the model results.
- If the dataset is the synthetic stand-in, these plots describe generated data, not real engines.